#### Simple Gen AI APP Using Langchain

In [125]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [126]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

In [127]:
loader=WebBaseLoader("https://docs.langchain.com/oss/python/learn")
loader

In [128]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/learn', 'title': 'Learn - Docs by LangChain', 'description': 'Tutorials, conceptual guides, and resources to help you get started.', 'language': 'en'}, page_content='Learn - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLearnDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesLangChain AcademyCase studiesGet helpOn this pageUse casesDeep AgentsLangChainLangGraphMulti-agentConceptual overviewsAdditional resourcesLearnCopy pageTutorials, conceptual guides, and resources to help you get started.Copy pageIn the Learn section of the documentation, you’ll find a collection of tutorials, conceptual overviews, and additional

In [129]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [130]:
documents

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/learn', 'title': 'Learn - Docs by LangChain', 'description': 'Tutorials, conceptual guides, and resources to help you get started.', 'language': 'en'}, page_content='Learn - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLearnDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesLangChain AcademyCase studiesGet helpOn this pageUse casesDeep AgentsLangChainLangGraphMulti-agentConceptual overviewsAdditional resourcesLearnCopy pageTutorials, conceptual guides, and resources to help you get started.Copy pageIn the Learn section of the documentation, you’ll find a collection of tutorials, conceptual overviews, and additional

In [131]:
#from langchain_openai import OpenAIEmbeddings
# embeddings=OpenAIEmbeddings()
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")

In [132]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [133]:
vectorstoredb

In [134]:
## Query From a vector db
query="Many of the applications you build with LangChain will contain multiple steps with multiple invocations of LLM calls. "
result=vectorstoredb.similarity_search(query)
result[0].page_content

'\u200bConceptual overviews\nThese guides explain the core concepts and APIs underlying LangChain and LangGraph.\nMemoryUnderstand persistence of interactions within and across threads.\nContext engineeringLearn methods for providing AI applications the right information and tools to accomplish a task.\nGraph APIExplore LangGraph’s declarative graph-building API.\nFunctional APIBuild agents as a single function.\n\u200bAdditional resources\nLangChain AcademyCourses and exercises to level up your LangChain skills.\nCase StudiesSee how teams are using LangChain and LangGraph in production.'

In [135]:
# from langchain_openai import ChatOpenAI
# llm=ChatOpenAI(model="gpt-4o")
from langchain_ollama import ChatOllama
llm = ChatOllama(model="Phi4")

In [136]:
## Retrieval Chain, Document chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOllama(model='Phi4')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [137]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

"Based on the provided context, LangSmith has two specific usage limits: total traces and extended traces. These limits are associated with the metrics tracked on their usage graph. The context does not provide further details about what these terms specifically entail or how they function within LangSmith's system."

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [138]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [139]:
retriever=vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [140]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000210E6829B50>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            | 

In [147]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"One of the most powerful applications enabled by LLMs is sophisticated question-answering (Q&A) chatbots"})
response['answer']

'The provided context is a structured documentation outline from the "Learn" section of LangChain\'s documentation. It offers guidance on building various types of agents using different frameworks like Deep Agents, LangChain, and LangGraph. Here are some key points based on the context:\n\n1. **Deep Agents**:\n   - Built-in functionality for managing context, a virtual filesystem, and other agent requirements.\n   - Tutorials include building data analysis agents that can send reports to Slack.\n\n2. **LangChain**:\n   - Provides simple implementations for use cases like semantic search engines, RAG agents, SQL agents, and voice agents.\n   - Focuses on ease of getting started with basic functionality.\n\n3. **LangGraph**:\n   - Utilizes LangGraph primitives for agent implementation, allowing for more customization.\n   - Offers guides to build custom RAG and SQL agents directly in LangGraph for fine-grained control or maximum flexibility.\n\n4. **Multi-agent Systems**:\n   - Demonstr

In [148]:

response

{'input': 'One of the most powerful applications enabled by LLMs is sophisticated question-answering (Q&A) chatbots',
 'context': [Document(id='209ef561-cf9c-460c-a5ac-07da543cdcbc', metadata={'source': 'https://docs.langchain.com/oss/python/learn', 'title': 'Learn - Docs by LangChain', 'description': 'Tutorials, conceptual guides, and resources to help you get started.', 'language': 'en'}, page_content='\u200bUse cases\nBelow are tutorials for common use cases, organized by framework.\n\u200bDeep Agents\nDeep agents include built-in functionality for managing context, a virtual filesystem, and other common agent requirements.\nData analysisBuild a data analysis agent that sends reports to Slack.\n\u200bLangChain\nLangChain agent implementations make it easy to get started for simple use cases.\nSemantic SearchBuild a semantic search engine over a PDF with LangChain components.\nRAG AgentCreate a Retrieval Augmented Generation (RAG) agent.\nSQL AgentBuild a SQL agent to interact with d

In [149]:
response['context']

[Document(id='209ef561-cf9c-460c-a5ac-07da543cdcbc', metadata={'source': 'https://docs.langchain.com/oss/python/learn', 'title': 'Learn - Docs by LangChain', 'description': 'Tutorials, conceptual guides, and resources to help you get started.', 'language': 'en'}, page_content='\u200bUse cases\nBelow are tutorials for common use cases, organized by framework.\n\u200bDeep Agents\nDeep agents include built-in functionality for managing context, a virtual filesystem, and other common agent requirements.\nData analysisBuild a data analysis agent that sends reports to Slack.\n\u200bLangChain\nLangChain agent implementations make it easy to get started for simple use cases.\nSemantic SearchBuild a semantic search engine over a PDF with LangChain components.\nRAG AgentCreate a Retrieval Augmented Generation (RAG) agent.\nSQL AgentBuild a SQL agent to interact with databases with human-in-the-loop review.\nVoice AgentBuild an agent you can speak and listen to.\n\u200bLangGraph\nLangChain’s agen